# Pandas Operations on Retail Sales Data

**Google Colab practice notebook | Uses the same eight CSV files as the Excel sales assignment**

This notebook demonstrates the requested Pandas operations in Parts 4–12 using the supplied customers, employees, order items, orders, products, returns, stores, and targets datasets. It keeps the original tables available and builds a joined order-item analysis table for examples. Run cells from top to bottom.

> Upload the eight CSV files when prompted. The separate `Sample - Superstore.csv` is not used in this notebook.


## 1. Setup and upload the eight CSV files

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

uploaded = files.upload()  # Select the eight CSV files together
required_files = [
    "customers.csv", "employees.csv", "order_items.csv", "orders.csv",
    "products.csv", "returns.csv", "stores.csv", "targets.csv"
]
missing_files = [name for name in required_files if name not in uploaded]
if missing_files:
    raise FileNotFoundError(f"Upload these files and rerun this cell: {missing_files}")

customers = pd.read_csv("customers.csv", encoding="utf-8-sig")
employees = pd.read_csv("employees.csv", encoding="utf-8-sig")
order_items = pd.read_csv("order_items.csv", encoding="utf-8-sig")
orders = pd.read_csv("orders.csv", encoding="utf-8-sig")
products = pd.read_csv("products.csv", encoding="utf-8-sig")
returns = pd.read_csv("returns.csv", encoding="utf-8-sig")
stores = pd.read_csv("stores.csv", encoding="utf-8-sig")
targets = pd.read_csv("targets.csv", encoding="utf-8-sig")

print("Loaded eight CSV files.")
for name, frame in {
    "customers": customers, "employees": employees, "order_items": order_items,
    "orders": orders, "products": products, "returns": returns,
    "stores": stores, "targets": targets
}.items():
    print(f"{name:12s} {frame.shape}")


## 2. Inspect the source tables

Check the fields and sample rows before joining or cleaning. Keep the original DataFrames unchanged so later examples can compare raw and cleaned values.

In [ ]:
display(orders.head())
display(order_items.head())
display(products.head())
print("Orders columns:", orders.columns.tolist())
print("Order item columns:", order_items.columns.tolist())


## 3. Build an order-item analysis DataFrame

Each row remains one product line in one order. Revenue applies the line discount; profit subtracts quantity times unit cost. This joined table is for demonstrations, while the original tables remain available for joins and table-specific operations.

In [ ]:
sales = order_items.merge(
    products[["ProductID", "ProductName", "Category", "Price", "Cost"]],
    on="ProductID", how="left", validate="many_to_one"
)
sales = sales.merge(orders, on="OrderID", how="left", validate="many_to_one")
sales = sales.merge(
    customers[["CustomerID", "Name", "City", "CustomerSegment"]]
        .rename(columns={"Name": "CustomerName"}),
    on="CustomerID", how="left", validate="many_to_one"
)
sales = sales.merge(
    employees[["EmployeeID", "Name"]].rename(columns={"Name": "EmployeeName"}),
    on="EmployeeID", how="left", validate="many_to_one"
)
sales = sales.merge(stores[["StoreID", "StoreName"]], on="StoreID", how="left", validate="many_to_one")
sales["Revenue"] = sales["Quantity"] * sales["Price"] * (1 - sales["Discount"])
sales["Profit"] = sales["Revenue"] - sales["Quantity"] * sales["Cost"]
sales["ProfitMargin"] = sales["Profit"].div(sales["Revenue"].replace(0, np.nan))

print("Joined order-item rows:", len(sales))
display(sales.head())


## Part 4 — Selecting and filtering data

`DataFrame.filter()` selects labels (such as columns); conditional masks and `.loc[]` are used to filter rows.

In [ ]:
# Selecting one column (returns a Series)
product_names = sales["ProductName"]
display(product_names.head())

# Selecting multiple columns (returns a DataFrame)
display(sales[["OrderID", "ProductName", "Category", "Revenue"]].head())

# loc[] is label-based; this DataFrame has a default integer index
display(sales.loc[0:4, ["OrderID", "ProductName", "Revenue"]])

# iloc[] is position-based
display(sales.iloc[0:5, 0:5])

# Conditional filtering
south_sales = sales.loc[sales["Region"].astype("string").str.strip().str.casefold().eq("south")]
display(south_sales.head())

# Multiple conditions: AND, OR, and NOT
and_filter = sales.loc[(sales["Quantity"] >= 3) & (sales["Revenue"] >= 10000)]
or_filter = sales.loc[(sales["SalesChannel"] == "Online") | (sales["SalesChannel"] == "Phone")]
not_filter = sales.loc[~sales["SalesChannel"].eq("Online")]
print("AND rows:", len(and_filter), "OR rows:", len(or_filter), "NOT-online rows:", len(not_filter))

# isin() and between()
selected_regions = sales.loc[sales["Region"].astype("string").str.strip().str.casefold().isin(["north", "west"])]
revenue_band = sales.loc[sales["Revenue"].between(10000, 50000, inclusive="both")]
print("Selected region rows:", len(selected_regions), "Revenue-band rows:", len(revenue_band))

# query() for readable conditions (backticks allow column names with spaces, if present)
query_result = sales.query("Quantity >= 3 and Revenue > 10000")
display(query_result.head())

# filter() selects columns by labels or matching text in the labels
selected_columns = sales.filter(items=["OrderID", "EmployeeName", "ProductName", "Revenue"])
name_columns = sales.filter(like="Name", axis="columns")
display(selected_columns.head())
print("Name-related columns:", name_columns.columns.tolist())

# select_dtypes() selects columns by data type
numeric_columns = sales.select_dtypes(include="number")
display(numeric_columns.head())


## Part 5 — Data cleaning and preprocessing

These examples use copies where an operation could change values. Review missingness and duplicate keys before deciding whether rows should actually be removed.

In [ ]:
# isnull() and isna() are equivalent checks; notnull() marks observed values
missing_isnull = sales.isnull().sum().sort_values(ascending=False)
missing_isna = sales.isna().sum().sort_values(ascending=False)
observed_order_ids = sales["OrderID"].notnull()
print("Missing values by column:")
display(missing_isna[missing_isna.gt(0)])
print("Rows with an OrderID:", int(observed_order_ids.sum()))

# Demonstration copy with a few explicit missing values (does not alter source files)
missing_demo = customers[["CustomerID", "City", "CustomerSegment"]].head(8).copy()
missing_demo.loc[missing_demo.index[1], "City"] = pd.NA
missing_demo.loc[missing_demo.index[3], "CustomerSegment"] = pd.NA
print("Before dropna:")
display(missing_demo)
print("dropna on required fields:")
display(missing_demo.dropna(subset=["CustomerID", "City"]))
print("fillna on a copy:")
display(missing_demo.fillna({"City": "Unknown", "CustomerSegment": "Unspecified"}))
print("Forward fill / backward fill examples:")
display(missing_demo.ffill().bfill())

# duplicated() and drop_duplicates()
print("Duplicate OrderID rows in orders:", int(orders["OrderID"].duplicated().sum()))
orders_unique_demo = orders.drop_duplicates(subset="OrderID", keep="first")
print("Order rows after key-based drop_duplicates:", len(orders_unique_demo))

# replace() and text normalization for common return-reason case variants
returns_clean = returns.copy()
returns_clean["Reason"] = returns_clean["Reason"].astype("string").str.strip().str.title()
returns_clean["Reason"] = returns_clean["Reason"].replace({"Not As Expected": "Not as Expected"})
print("Normalized return reasons:")
display(returns_clean["Reason"].value_counts(dropna=False))

# astype(), convert_dtypes(), rename(), and drop() on demonstration copies
products_typed = products.copy()
products_typed["ProductID"] = products_typed["ProductID"].astype("string")
products_typed["Price"] = pd.to_numeric(products_typed["Price"], errors="coerce").astype("Float64")
products_typed = products_typed.convert_dtypes()
products_renamed = products_typed.rename(columns={"ProductName": "Product_Name"})
products_without_supplier = products_renamed.drop(columns=["Supplier"], errors="ignore")
print(products_without_supplier.dtypes)

# IQR outlier review: flag unusual Revenue values; do not delete automatically
q1, q3 = sales["Revenue"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
sales["RevenueOutlierIQR"] = ~sales["Revenue"].between(lower_bound, upper_bound)
print(f"Revenue IQR bounds: {lower_bound:,.2f} to {upper_bound:,.2f}")
print("Rows flagged for review:", int(sales["RevenueOutlierIQR"].sum()))


## Part 6 — Data manipulation and transformation

Use vectorized arithmetic for calculations across columns. `apply()` and `map()` are included to demonstrate the requested methods.

In [ ]:
transform_demo = sales[["OrderID", "ProductName", "Category", "Quantity", "Price", "Cost", "Discount", "Revenue", "Profit"]].head(12).copy()

# Adding a new column and updating an existing column
transform_demo["GrossRevenue"] = transform_demo["Quantity"] * transform_demo["Price"]
transform_demo["Category"] = transform_demo["Category"].astype("string").str.strip()

# insert() places a new column at the requested position
transform_demo.insert(1, "LineLabel", "Order item")

# apply() applies a function to a Series; axis=1 applies across each row
transform_demo["RevenueBand"] = transform_demo["Revenue"].apply(lambda amount: "High" if amount >= 10000 else "Standard")
transform_demo["LineSummary"] = transform_demo.apply(
    lambda row: f"{row['ProductName']} — {row['Quantity']} units", axis=1
)

# map() maps values in one Series
category_group = {"Books": "General", "Beauty": "Lifestyle", "Clothing": "Lifestyle"}
transform_demo["CategoryGroup"] = transform_demo["Category"].map(category_group).fillna("Other")

# DataFrame.map() applies a scalar function to each element; available in pandas 2.1+
text_demo = transform_demo[["ProductName", "Category"]].copy()
if hasattr(text_demo, "map"):
    text_demo = text_demo.map(lambda value: value.strip() if isinstance(value, str) else value)
else:
    print("DataFrame.map() requires pandas 2.1 or later; upgrade pandas in Colab if needed.")

# assign() returns a DataFrame with new/updated columns
assigned = transform_demo.assign(ProfitMargin=lambda frame: frame["Profit"] / frame["Revenue"])

# where() keeps values where the condition is True; mask() replaces where True
transform_demo["RevenueAtLeast10000"] = transform_demo["Revenue"].where(transform_demo["Revenue"] >= 10000, 0)
transform_demo["LowMarginFlag"] = transform_demo["Profit"].div(transform_demo["Revenue"]).lt(0.10)
transform_demo["ProfitMaskedIfNegative"] = transform_demo["Profit"].mask(transform_demo["Profit"] < 0)

# Vectorized operations (no row-wise loop)
transform_demo["DiscountedPricePerUnit"] = transform_demo["Price"] * (1 - transform_demo["Discount"])

# Deleting columns on a copy
transform_demo_without_label = transform_demo.drop(columns=["LineLabel"])
display(transform_demo_without_label.head())
display(assigned[["ProductName", "Revenue", "Profit", "ProfitMargin"]].head())


### Part 7 — Sorting and ranking

In [ ]:
# sort_values() sorts by one or more columns
sorted_sales = sales.sort_values(["Revenue", "OrderDate"], ascending=[False, True], na_position="last")
display(sorted_sales[["OrderID", "ProductName", "Revenue"]].head(10))

# sort_index() sorts index labels
sorted_index_demo = sales.set_index("OrderItemID").sort_index()
display(sorted_index_demo[["OrderID", "ProductName"]].head())

# rank() ranks rows by Revenue
ranked_sales = sales[["OrderID", "ProductName", "Revenue"]].copy()
ranked_sales["RevenueRank"] = ranked_sales["Revenue"].rank(method="dense", ascending=False)
display(ranked_sales.sort_values("RevenueRank").head(10))

# nlargest() and nsmallest()
display(sales.nlargest(10, "Revenue")[["OrderID", "ProductName", "Revenue", "Profit"]])
display(sales.nsmallest(10, "Profit")[["OrderID", "ProductName", "Revenue", "Profit"]])


### Part 8 — GroupBy and aggregation

The examples summarize order-item revenue, profit, and quantity by product category, employee, and channel.

In [ ]:
# Grouping by one column
revenue_by_category = sales.groupby("Category", dropna=False)["Revenue"].sum().sort_values(ascending=False)
display(revenue_by_category)

# Grouping by multiple columns
revenue_by_region_channel = sales.groupby(["Region", "SalesChannel"], dropna=False)["Revenue"].sum()
display(revenue_by_region_channel)

# Aggregation functions and agg()
category_summary = sales.groupby("Category", dropna=False).agg({
    "Revenue": ["sum", "mean", "min", "max"],
    "Profit": ["sum", "mean"],
    "Quantity": ["sum", "count"]
})
display(category_summary)

# Named aggregation creates reader-friendly output column names
employee_summary = sales.groupby("EmployeeName", dropna=False).agg(
    UnitsSold=("Quantity", "sum"),
    Revenue=("Revenue", "sum"),
    Profit=("Profit", "sum"),
    OrderCount=("OrderID", "nunique")
).sort_values("Revenue", ascending=False)
display(employee_summary.head(10))

# transform() returns a result aligned to the original rows
sales["CategoryRevenueTotal"] = sales.groupby("Category")["Revenue"].transform("sum")
sales["ShareOfCategoryRevenue"] = sales["Revenue"] / sales["CategoryRevenueTotal"]

# GroupBy filter(): keep categories with at least $20 million revenue
large_categories = sales.groupby("Category").filter(lambda group: group["Revenue"].sum() >= 20_000_000)
print("Rows in categories with at least $20m revenue:", len(large_categories))

# Custom aggregation and reset_index()
category_spread = (sales.groupby("Category")["Revenue"]
                   .agg(lambda values: values.max() - values.min())
                   .reset_index(name="RevenueRange"))
display(category_spread.sort_values("RevenueRange", ascending=False))


### Part 9 — Combining datasets

`concat()` stacks or aligns tables. `merge()` combines columns by key; `join()` combines tables by index. Examples use the supplied data, with small previews for readability.

In [ ]:
# concat(): stack rows and align columns side-by-side
concat_rows = pd.concat([orders.head(3), orders.iloc[3:6]], ignore_index=True)
concat_columns = pd.concat([customers[["CustomerID"]].head(), customers[["City"]].head()], axis=1)
print("Rows after vertical concat:", len(concat_rows))
display(concat_columns)

# Inner, left, right, and outer joins between order items and products
items_preview = order_items.head(100)
product_preview = products.head(80)
inner_join = items_preview.merge(product_preview, on="ProductID", how="inner")
left_join = items_preview.merge(product_preview, on="ProductID", how="left", indicator=True)
right_join = items_preview.merge(product_preview, on="ProductID", how="right")
outer_join = items_preview.merge(product_preview, on="ProductID", how="outer", indicator=True)
print("Join row counts:", len(inner_join), len(left_join), len(right_join), len(outer_join))

# Merge using multiple columns: illustrative key match on customer and region
left_multi = orders[["CustomerID", "Region"]].drop_duplicates().head(20)
right_multi = orders[["CustomerID", "Region", "SalesChannel"]].drop_duplicates().head(30)
multiple_key_merge = left_multi.merge(right_multi, on=["CustomerID", "Region"], how="left")
display(multiple_key_merge.head())

# Self join: pair employees who share a region (small preview)
peer_demo = employees[["EmployeeID", "Name", "Region"]].head(12)
self_join = peer_demo.merge(peer_demo, on="Region", suffixes=("_A", "_B"))
self_join = self_join[self_join["EmployeeID_A"] != self_join["EmployeeID_B"]]
display(self_join.head())

# join() uses indexes rather than an explicit on= key
customer_index = customers.set_index("CustomerID")[["City", "CustomerSegment"]]
order_count_index = orders.groupby("CustomerID").size().rename("OrderCount")
joined_by_index = customer_index.join(order_count_index, how="left").fillna({"OrderCount": 0})
display(joined_by_index.head())


### Part 10 — Date and time operations

Invalid source dates are coerced to `NaT` and counted. Monthly summaries use valid dates only; this keeps invalid dates from being assigned to an invented month.

In [ ]:
# to_datetime() and invalid date count
orders_dates = orders.copy()
orders_dates["OrderDate"] = pd.to_datetime(orders_dates["OrderDate"], errors="coerce")
invalid_order_dates = int(orders_dates["OrderDate"].isna().sum())
print("Invalid or missing OrderDate values:", invalid_order_dates)

# Extract year, month, day, and weekday
orders_dates["Year"] = orders_dates["OrderDate"].dt.year
orders_dates["Month"] = orders_dates["OrderDate"].dt.month
orders_dates["Day"] = orders_dates["OrderDate"].dt.day
orders_dates["Weekday"] = orders_dates["OrderDate"].dt.day_name()
display(orders_dates[["OrderID", "OrderDate", "Year", "Month", "Day", "Weekday"]].head())

# Date filtering and date differences from customer signup
customer_dates = customers.copy()
customer_dates["SignupDate"] = pd.to_datetime(customer_dates["SignupDate"], errors="coerce")
orders_customer_dates = orders_dates.merge(customer_dates[["CustomerID", "SignupDate"]], on="CustomerID", how="left")
period_filter = orders_customer_dates.loc[
    orders_customer_dates["OrderDate"].between("2024-01-01", "2024-12-31", inclusive="both")
]
orders_customer_dates["DaysFromSignupToOrder"] = (
    orders_customer_dates["OrderDate"] - orders_customer_dates["SignupDate"]
).dt.days
display(period_filter.head())

# date_range() creates a continuous calendar
date_calendar = pd.date_range(start="2024-01-01", end="2024-01-07", freq="D")
print(date_calendar)

# Time zone example: source dates are date-only; demonstrate an explicit Asia/Kolkata assumption
sample_local = pd.to_datetime(orders_dates["OrderDate"].dropna().head(3))
sample_local = sample_local.dt.tz_localize("Asia/Kolkata")
sample_utc = sample_local.dt.tz_convert("UTC")
print("Localized timestamps:")
display(sample_local)
print("Converted to UTC:")
display(sample_utc)

# resample(), rolling(), shift(), pct_change(), and diff() on monthly revenue
sales_dates = sales.copy()
sales_dates["OrderDate"] = pd.to_datetime(sales_dates["OrderDate"], errors="coerce")
monthly_revenue = (sales_dates.dropna(subset=["OrderDate"])
                   .set_index("OrderDate")["Revenue"].resample("MS").sum())
monthly_trend = monthly_revenue.to_frame("Revenue")
monthly_trend["ThreeMonthRollingAverage"] = monthly_trend["Revenue"].rolling(3, min_periods=1).mean()
monthly_trend["PreviousMonthRevenue"] = monthly_trend["Revenue"].shift(1)
monthly_trend["RevenuePctChange"] = monthly_trend["Revenue"].pct_change()
monthly_trend["RevenueDifference"] = monthly_trend["Revenue"].diff()
display(monthly_trend.head(12))


### Part 11 — String operations and regular expressions

The following examples use product names, categories, customer names, emails, and return reasons. They create derived values and preserve source columns.

In [ ]:
string_demo = sales[["ProductName", "Category", "CustomerName"]].copy()
string_demo["ProductLower"] = string_demo["ProductName"].str.lower()
string_demo["CategoryUpper"] = string_demo["Category"].str.upper()
string_demo["CustomerTitle"] = string_demo["CustomerName"].str.title()
string_demo["ProductStripped"] = string_demo["ProductName"].str.strip()
string_demo["ProductReplaced"] = string_demo["ProductName"].str.replace("Pro ", "Professional ", regex=False)
string_demo["HasBottle"] = string_demo["ProductName"].str.contains("bottle", case=False, na=False)
string_demo["StartsWithEco"] = string_demo["ProductName"].str.startswith("Eco", na=False)
string_demo["EndsWithThreeDigits"] = string_demo["ProductName"].str.contains(r"\d{3}$", regex=True, na=False)
string_demo["ProductWords"] = string_demo["ProductName"].str.split()
string_demo["ProductPrefix"] = string_demo["ProductName"].str.extract(r"^([A-Za-z]+)", expand=False)
string_demo["ProductNameLength"] = string_demo["ProductName"].str.len()
display(string_demo.head(10))

# String cleanup and regular expression example for return reasons
reason_demo = returns[["Reason"]].copy()
reason_demo["ReasonClean"] = reason_demo["Reason"].astype("string").str.strip().str.title()
reason_demo["ReasonWords"] = reason_demo["ReasonClean"].str.split()
reason_demo["ReasonToken"] = reason_demo["ReasonClean"].str.extract(r"^([A-Za-z]+)", expand=False)
display(reason_demo.head(10))


### Part 12 — Statistical analysis

Basic descriptive statistics summarize order-item measures. Correlation and covariance describe association, not cause.

In [ ]:
numeric = sales[["Quantity", "Price", "Cost", "Discount", "Revenue", "Profit", "ProfitMargin"]].apply(pd.to_numeric, errors="coerce")

# mean, median, mode, sum, count, min, max, standard deviation, variance, quantiles
statistics = pd.DataFrame({
    "mean": numeric.mean(),
    "median": numeric.median(),
    "mode": numeric.mode().iloc[0],
    "sum": numeric.sum(),
    "count": numeric.count(),
    "min": numeric.min(),
    "max": numeric.max(),
    "std": numeric.std(),
    "variance": numeric.var(),
    "25th_percentile": numeric.quantile(0.25),
    "75th_percentile": numeric.quantile(0.75)
})
display(statistics)

# corr() and cov() on numeric variables
print("Correlation matrix:")
display(numeric[["Quantity", "Price", "Discount", "Revenue", "Profit"]].corr())
print("Covariance matrix:")
display(numeric[["Quantity", "Price", "Discount", "Revenue", "Profit"]].cov())

# cumsum(), cumprod(), pct_change(), and diff() on chronological monthly revenue
monthly_stats = monthly_revenue.sort_index().to_frame("Revenue")
monthly_stats["CumulativeRevenue"] = monthly_stats["Revenue"].cumsum()
monthly_stats["MonthlyPctChange"] = monthly_stats["Revenue"].pct_change()
monthly_stats["MonthlyDifference"] = monthly_stats["Revenue"].diff()
monthly_stats["RevenueIndex"] = (1 + monthly_stats["MonthlyPctChange"].fillna(0)).cumprod()
display(monthly_stats.head(12))


## Sales performance assignment — Questions 1–21

This section calculates the business answers from the same eight CSVs. It runs after the setup and join cells above. Revenue and profit are calculated at the order-item level; order counts use distinct OrderID values. Invalid OrderDate values are excluded from monthly calculations and counted in the data-quality summary.

### Questions 1–4 — Company totals, orders, months, and regions

In [ ]:
# Prepare consistent analysis fields
analysis = sales.copy()
analysis["OrderDate"] = pd.to_datetime(analysis["OrderDate"], errors="coerce")
analysis["Region"] = analysis["Region"].astype("string").str.strip().str.title()
analysis["SalesChannel"] = analysis["SalesChannel"].astype("string").str.strip().str.title()
analysis["CustomerSegment"] = analysis["CustomerSegment"].astype("string").str.strip().str.title()

# Q1: total revenue, total profit, and total quantity sold
company_totals = pd.DataFrame({
    "Metric": ["Total Revenue", "Total Profit", "Total Quantity Sold"],
    "Value": [analysis["Revenue"].sum(), analysis["Profit"].sum(), analysis["Quantity"].sum()]
})
display(company_totals)

# Q2: top 10 highest-revenue orders, with order/customer/region
order_summary = (analysis.groupby(["OrderID", "CustomerID", "Region"], dropna=False)
                 .agg(Revenue=("Revenue", "sum"), Profit=("Profit", "sum"), Quantity=("Quantity", "sum"))
                 .reset_index())
top_10_orders = order_summary.nlargest(10, "Revenue")
display(top_10_orders)

# Q3: monthly revenue and highest/lowest month; excludes invalid dates
valid_date_sales = analysis.dropna(subset=["OrderDate"]).copy()
valid_date_sales["Month"] = valid_date_sales["OrderDate"].dt.to_period("M").dt.to_timestamp()
monthly_revenue_summary = (valid_date_sales.groupby("Month", as_index=False)
                           .agg(Revenue=("Revenue", "sum"))
                           .sort_values("Month"))
highest_month = monthly_revenue_summary.loc[monthly_revenue_summary["Revenue"].idxmax()]
lowest_month = monthly_revenue_summary.loc[monthly_revenue_summary["Revenue"].idxmin()]
print("Highest-revenue month:")
display(highest_month.to_frame().T)
print("Lowest-revenue month:")
display(lowest_month.to_frame().T)

# Q4: top five regions (if fewer than five exist, all are shown) and revenue contribution
region_revenue = (analysis.groupby("Region", dropna=False)
                  .agg(Revenue=("Revenue", "sum"))
                  .sort_values("Revenue", ascending=False)
                  .reset_index())
region_revenue["RevenueSharePct"] = region_revenue["Revenue"] / analysis["Revenue"].sum() * 100
top_5_regions = region_revenue.head(5)
display(top_5_regions)


### Questions 5–7 — Product revenue, profit, and margins

In [ ]:
# Q5: top 10 products by revenue with category, quantity, revenue, and profit
product_summary = (analysis.groupby(["ProductID", "ProductName", "Category"], dropna=False)
                   .agg(QuantitySold=("Quantity", "sum"), Revenue=("Revenue", "sum"), Profit=("Profit", "sum"))
                   .reset_index())
product_summary["ProfitMarginPct"] = product_summary["Profit"].div(product_summary["Revenue"].replace(0, np.nan)) * 100
top_10_products = product_summary.nlargest(10, "Revenue")
display(top_10_products[["ProductName", "Category", "QuantitySold", "Revenue", "Profit"]])

# Q6: five most profitable categories and five products with lowest profit
category_profit = (analysis.groupby("Category", dropna=False)["Profit"]
                   .sum().sort_values(ascending=False).rename("Profit").reset_index())
top_5_categories_by_profit = category_profit.head(5)
lowest_5_profit_products = product_summary.nsmallest(5, "Profit")
print("Top product categories by profit:")
display(top_5_categories_by_profit)
print("Five products with the lowest profit:")
display(lowest_5_profit_products[["ProductName", "Category", "QuantitySold", "Revenue", "Profit"]])

# Q7: product-level profit margins; identify margins below 10%
products_below_10_margin = product_summary.loc[product_summary["ProfitMarginPct"] < 10].sort_values("ProfitMarginPct")
print("Products with profit margin below 10%:", len(products_below_10_margin))
display(products_below_10_margin[["ProductName", "Category", "Revenue", "Profit", "ProfitMarginPct"]])


### Questions 8–11 — Employee performance and employee-product sales

In [ ]:
# Q8–9: employee order count, quantity, revenue, profit, top 10, and leaders
employee_sales = (analysis.groupby(["EmployeeID", "EmployeeName"], dropna=False)
                  .agg(OrderCount=("OrderID", "nunique"), QuantitySold=("Quantity", "sum"),
                       Revenue=("Revenue", "sum"), Profit=("Profit", "sum"))
                  .reset_index())
# Include employees with no sales as zero rather than silently omitting them
employee_performance = employees[["EmployeeID", "Name"]].rename(columns={"Name": "EmployeeName"}).merge(
    employee_sales, on=["EmployeeID", "EmployeeName"], how="left"
)
for col in ["OrderCount", "QuantitySold", "Revenue", "Profit"]:
    employee_performance[col] = employee_performance[col].fillna(0)
top_10_employees = employee_performance.nlargest(10, "Revenue")
highest_revenue_employee = employee_performance.loc[employee_performance["Revenue"].idxmax()]
highest_order_count_employee = employee_performance.loc[employee_performance["OrderCount"].idxmax()]
display(top_10_employees.sort_values("Revenue", ascending=False))
print("Highest revenue employee:")
display(highest_revenue_employee.to_frame().T)
print("Most orders handled:")
display(highest_order_count_employee.to_frame().T)

# Q10: products sold by each employee; full output can be large
employee_product_sales = (analysis.groupby(["EmployeeName", "ProductName", "Category"], dropna=False)
                          .agg(QuantitySold=("Quantity", "sum"), Revenue=("Revenue", "sum"), Profit=("Profit", "sum"))
                          .reset_index()
                          .sort_values(["EmployeeName", "Revenue"], ascending=[True, False]))
print("Employee-product combinations:", len(employee_product_sales))
display(employee_product_sales.head(20))

# Q11: best-performing employee in each product category by revenue
employee_category_revenue = (analysis.groupby(["Category", "EmployeeName"], dropna=False)
                             .agg(Revenue=("Revenue", "sum"))
                             .reset_index())
best_employee_by_category = (employee_category_revenue.sort_values(["Category", "Revenue"], ascending=[True, False])
                             .groupby("Category", dropna=False).head(1)
                             .sort_values("Category"))
display(best_employee_by_category)


### Questions 12–14 — Customer analysis

In [ ]:
# Q12–13: top 10 customers and revenue by segment
customer_order_sales = (analysis.groupby("CustomerID", dropna=False)
                        .agg(NumberOfOrders=("OrderID", "nunique"), TotalRevenue=("Revenue", "sum"))
                        .reset_index())
customer_summary = customers[["CustomerID", "Name", "City", "CustomerSegment"]].rename(
    columns={"Name": "CustomerName"}
).merge(customer_order_sales, on="CustomerID", how="left")
customer_summary["NumberOfOrders"] = customer_summary["NumberOfOrders"].fillna(0).astype(int)
customer_summary["TotalRevenue"] = customer_summary["TotalRevenue"].fillna(0)
top_10_customers = customer_summary.nlargest(10, "TotalRevenue")
display(top_10_customers[["CustomerName", "City", "CustomerSegment", "NumberOfOrders", "TotalRevenue"]])

segment_revenue = (customer_summary.groupby("CustomerSegment", dropna=False)["TotalRevenue"]
                   .sum().sort_values(ascending=False).rename("Revenue").reset_index())
display(segment_revenue)

# Q14: customers registered but never present in orders
ordered_customer_ids = orders["CustomerID"].dropna().unique()
customers_without_orders = customers.loc[~customers["CustomerID"].isin(ordered_customer_ids)].copy()
print("Registered customers with no orders:", len(customers_without_orders))
display(customers_without_orders.head(20))


### Questions 15–16 — Store, region, and sales channel analysis

In [ ]:
# Q15: store-wise revenue, including stores with no recorded sales
store_revenue = (analysis.groupby(["StoreID", "StoreName"], dropna=False)["Revenue"]
                 .sum().rename("Revenue").reset_index())
store_performance = stores[["StoreID", "StoreName", "City", "Region", "StoreType"]].merge(
    store_revenue, on=["StoreID", "StoreName"], how="left"
)
store_performance["Revenue"] = store_performance["Revenue"].fillna(0)
store_performance = store_performance.sort_values("Revenue", ascending=False)
print("Highest-performing store:")
display(store_performance.head(1))
print("Lowest-performing store:")
display(store_performance.tail(1))
display(store_performance)

# Q16: compare Online, In-Store, and Phone sales
channel_revenue = (analysis.groupby("SalesChannel", dropna=False)["Revenue"]
                   .sum().sort_values(ascending=False).rename("Revenue").reset_index())
display(channel_revenue)


### Questions 17–19 — Returns and targets

In [ ]:
# Q17: total returned orders and most common normalized return reason
returns_analysis = returns.copy()
returns_analysis["ReasonClean"] = returns_analysis["Reason"].astype("string").str.strip().str.title()
returns_analysis["ReasonClean"] = returns_analysis["ReasonClean"].replace({"Not As Expected": "Not as Expected"})
returned_order_count = returns_analysis["OrderID"].nunique()
reason_counts = returns_analysis["ReasonClean"].value_counts(dropna=False).rename_axis("Reason").reset_index(name="ReturnRecords")
most_common_reason = reason_counts.dropna(subset=["Reason"]).iloc[0] if not reason_counts.dropna(subset=["Reason"]).empty else None
print("Unique returned orders:", returned_order_count)
print("Most common return reason:")
display(most_common_reason.to_frame().T if most_common_reason is not None else "No reason recorded")
print("Reason counts, including missing reasons:")
display(reason_counts)

# Q18: regional return rate = unique returned orders / unique total orders
order_region = orders[["OrderID", "Region"]].drop_duplicates("OrderID").copy()
order_region["Region"] = order_region["Region"].astype("string").str.strip().str.title()
returned_with_region = returns_analysis[["OrderID"]].drop_duplicates().merge(order_region, on="OrderID", how="left")
region_order_counts = order_region.groupby("Region")["OrderID"].nunique().rename("TotalOrders")
region_return_counts = returned_with_region.groupby("Region")["OrderID"].nunique().rename("ReturnedOrders")
regional_return_rates = pd.concat([region_order_counts, region_return_counts], axis=1).fillna(0).reset_index()
regional_return_rates["ReturnRatePct"] = regional_return_rates["ReturnedOrders"] / regional_return_rates["TotalOrders"] * 100
regional_return_rates = regional_return_rates.sort_values("ReturnRatePct", ascending=False)
display(regional_return_rates)

# Q19: actual revenue versus monthly regional targets
actual_monthly_region = (valid_date_sales.assign(Month=valid_date_sales["OrderDate"].dt.to_period("M").dt.to_timestamp())
                         .groupby(["Month", "Region"], dropna=False)["Revenue"]
                         .sum().rename("ActualRevenue").reset_index())
target_comparison = targets.copy()
target_comparison["Month"] = pd.to_datetime(target_comparison["Month"], errors="coerce").dt.to_period("M").dt.to_timestamp()
target_comparison["Region"] = target_comparison["Region"].astype("string").str.strip().str.title()
target_comparison = target_comparison.merge(actual_monthly_region, on=["Month", "Region"], how="left")
target_comparison["ActualRevenue"] = target_comparison["ActualRevenue"].fillna(0)
target_comparison["Variance"] = target_comparison["ActualRevenue"] - target_comparison["SalesTarget"]
target_comparison["AchievementPct"] = target_comparison["ActualRevenue"].div(target_comparison["SalesTarget"].replace(0, np.nan)) * 100
target_comparison["Status"] = np.where(target_comparison["ActualRevenue"] >= target_comparison["SalesTarget"], "Achieved", "Below Target")
display(target_comparison.head(20))

regional_target_summary = (target_comparison.groupby("Region", dropna=False)
                           .agg(ActualRevenue=("ActualRevenue", "sum"), SalesTarget=("SalesTarget", "sum"))
                           .reset_index())
regional_target_summary["AchievementPct"] = regional_target_summary["ActualRevenue"].div(regional_target_summary["SalesTarget"].replace(0, np.nan)) * 100
regional_target_summary["OverallStatus"] = np.where(regional_target_summary["ActualRevenue"] >= regional_target_summary["SalesTarget"], "Achieved", "Below Target")
display(regional_target_summary.sort_values("AchievementPct", ascending=False))


### Questions 20–21 — Five insights and recommendations

In [ ]:
# Build five evidence-based insight statements from the computed summaries
best_region = region_revenue.iloc[0]
lowest_region = region_revenue.iloc[-1]
highest_return_region = regional_return_rates.iloc[0]
best_channel = channel_revenue.iloc[0]
best_product = product_summary.loc[product_summary["Profit"] > 0].sort_values("Revenue", ascending=False).iloc[0]
best_employee = employee_performance.sort_values("Revenue", ascending=False).iloc[0]
lowest_target_region = regional_target_summary.sort_values("AchievementPct").iloc[0]

business_insights = pd.DataFrame([
    {
        "Finding": "Regional revenue is uneven.",
        "Evidence": f"{best_region['Region']} leads at {best_region['Revenue']:,.2f}; {lowest_region['Region']} is lowest at {lowest_region['Revenue']:,.2f}.",
        "Recommendation": f"Protect {best_region['Region']}'s performance and investigate customer demand, coverage, and conversion in {lowest_region['Region']}."
    },
    {
        "Finding": "One region has the highest return rate.",
        "Evidence": f"{highest_return_region['Region']} has {highest_return_region['ReturnRatePct']:.2f}% returns ({int(highest_return_region['ReturnedOrders'])} of {int(highest_return_region['TotalOrders'])} orders).",
        "Recommendation": "Review the leading return reasons and check fulfillment, product quality, and delivery in that region."
    },
    {
        "Finding": "A high-revenue, profitable product is a promotion candidate.",
        "Evidence": f"{best_product['ProductName']} generated {best_product['Revenue']:,.2f} revenue and {best_product['Profit']:,.2f} profit.",
        "Recommendation": "Promote this product while tracking its margin and stock availability; avoid using revenue alone to choose promotions."
    },
    {
        "Finding": "One employee leads total revenue.",
        "Evidence": f"{best_employee['EmployeeName']} generated {best_employee['Revenue']:,.2f} revenue and handled {int(best_employee['OrderCount'])} orders.",
        "Recommendation": "Recognize this employee and share effective practices with the team."
    },
    {
        "Finding": "Regional target attainment varies.",
        "Evidence": f"{lowest_target_region['Region']} has the lowest overall achievement at {lowest_target_region['AchievementPct']:.2f}%; the highest-revenue channel is {best_channel['SalesChannel']}.",
        "Recommendation": "Review monthly shortfalls in the lowest-attainment region and support the best-performing channel where profitable."
    }
])
display(business_insights)

# Data-quality notes for the report
quality_notes = pd.DataFrame({
    "Check": ["Invalid order dates", "Return records with missing reason", "Distinct sales regions", "Distinct sales channels"],
    "Result": [
        int(orders["OrderDate"].pipe(pd.to_datetime, errors="coerce").isna().sum()),
        int(returns_analysis["ReasonClean"].isna().sum()),
        int(analysis["Region"].nunique(dropna=True)),
        int(analysis["SalesChannel"].nunique(dropna=True))
    ]
})
display(quality_notes)


### Export the assignment result tables

Save the full result tables as CSVs in the Colab runtime. Use the Files panel to download them, or uncomment the optional download lines.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("sales_assignment_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
result_tables = {
    "company_totals": company_totals,
    "top_10_orders": top_10_orders,
    "monthly_revenue": monthly_revenue_summary,
    "region_revenue": region_revenue,
    "top_10_products": top_10_products,
    "top_5_categories_by_profit": top_5_categories_by_profit,
    "lowest_5_profit_products": lowest_5_profit_products,
    "products_below_10_margin": products_below_10_margin,
    "employee_performance": employee_performance,
    "employee_product_sales": employee_product_sales,
    "best_employee_by_category": best_employee_by_category,
    "top_10_customers": top_10_customers,
    "segment_revenue": segment_revenue,
    "customers_without_orders": customers_without_orders,
    "store_performance": store_performance,
    "channel_revenue": channel_revenue,
    "return_reason_counts": reason_counts,
    "regional_return_rates": regional_return_rates,
    "target_comparison_monthly": target_comparison,
    "regional_target_summary": regional_target_summary,
    "business_insights": business_insights,
    "quality_notes": quality_notes,
}
for filename, frame in result_tables.items():
    frame.to_csv(OUTPUT_DIR / f"{filename}.csv", index=False)
print(f"Saved {len(result_tables)} result tables to {OUTPUT_DIR.resolve()}")
# Optional: download a specific file from Colab
# files.download(str(OUTPUT_DIR / "business_insights.csv"))


## Visualizations — charts for the sales analysis

Run this section after the joined `sales` DataFrame has been created. It produces graphs for monthly revenue, regional revenue, sales channels, top products, return reasons, and actual revenue versus targets. Invalid order dates are excluded from date-based charts.

In [ ]:
import matplotlib.pyplot as plt

# Prepare plotting fields without changing the original joined data
sales_plot = sales.copy()
sales_plot["OrderDate"] = pd.to_datetime(sales_plot["OrderDate"], errors="coerce")
sales_plot["RegionClean"] = sales_plot["Region"].astype("string").str.strip().str.title()
sales_plot["MonthStart"] = sales_plot["OrderDate"].dt.to_period("M").dt.to_timestamp()

# 1. Monthly revenue trend
monthly_revenue = (
    sales_plot.dropna(subset=["MonthStart"])
    .groupby("MonthStart")["Revenue"]
    .sum()
)
plt.figure(figsize=(11, 4))
plt.plot(monthly_revenue.index, monthly_revenue.values, marker="o", linewidth=2, color="#2364AA")
plt.title("Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 2. Revenue by region and by sales channel
region_chart = sales_plot.groupby("RegionClean")["Revenue"].sum().sort_values()
channel_chart = sales_plot.groupby("SalesChannel")["Revenue"].sum().sort_values()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
region_chart.plot(kind="barh", ax=axes[0], color="#2A9D8F", title="Revenue by Region")
channel_chart.plot(kind="bar", ax=axes[1], color="#E9C46A", title="Revenue by Sales Channel")
axes[0].set_xlabel("Revenue")
axes[0].set_ylabel("Region")
axes[1].set_xlabel("Sales Channel")
axes[1].set_ylabel("Revenue")
axes[1].tick_params(axis="x", rotation=20)
fig.tight_layout()
plt.show()

# 3. Top 10 products by revenue
top_products_chart = (sales_plot.groupby("ProductName")["Revenue"].sum()
                      .nlargest(10).sort_values())
plt.figure(figsize=(10, 5))
top_products_chart.plot(kind="barh", color="#F4A261")
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

# 4. Return reasons (standardize capitalization; missing reasons remain labeled)
returns_plot = returns.copy()
returns_plot["ReasonClean"] = returns_plot["Reason"].astype("string").str.strip().str.title()
returns_plot["ReasonClean"] = returns_plot["ReasonClean"].fillna("Missing reason")
reason_chart = returns_plot["ReasonClean"].value_counts().sort_values()
plt.figure(figsize=(9, 4))
reason_chart.plot(kind="barh", color="#E76F51")
plt.title("Returned Orders by Reason")
plt.xlabel("Return records")
plt.ylabel("Return reason")
plt.tight_layout()
plt.show()

# 5. Monthly actual revenue versus sales targets, by region
actual_monthly_region = (sales_plot.dropna(subset=["MonthStart"])
                         .groupby(["MonthStart", "RegionClean"], as_index=False)
                         .agg(ActualRevenue=("Revenue", "sum")))
targets_plot = targets.copy()
targets_plot["TargetMonth"] = pd.to_datetime(targets_plot["Month"], errors="coerce").dt.to_period("M").dt.to_timestamp()
targets_plot["RegionClean"] = targets_plot["Region"].astype("string").str.strip().str.title()
compare = targets_plot.merge(
    actual_monthly_region,
    left_on=["TargetMonth", "RegionClean"],
    right_on=["MonthStart", "RegionClean"],
    how="left"
)
compare["ActualRevenue"] = compare["ActualRevenue"].fillna(0)
fig, ax = plt.subplots(figsize=(12, 5))
for region, group in compare.groupby("RegionClean"):
    group = group.sort_values("TargetMonth")
    ax.plot(group["TargetMonth"], group["ActualRevenue"], label=f"{region} actual", linewidth=1.8)
    ax.plot(group["TargetMonth"], group["SalesTarget"], label=f"{region} target", linestyle="--", alpha=0.8)
ax.set_title("Monthly Actual Revenue vs Regional Sales Targets")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.25)
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
plt.show()


## Wrap-up and interpretation

- Keep original source DataFrames separate from cleaned or derived copies.
- Record how many invalid dates, missing values, and duplicate keys were found before deciding how to handle them.
- Revenue and profit are calculated at the order-item level. When summarizing orders, customers, or employees, aggregate the line items to the requested level first.
- The `Sample - Superstore.csv` file, if present, is a separate dataset and is intentionally not merged with these eight retail tables.
